In [ ]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GroupShuffleSplit, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, make_scorer
)
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

DATA_PATH = "Merged_final_IoT DDoS Dataset.csv"   
FEATURES  = ["Flow Duration", "Flow IAT Mean", "Tot Fwd Pkts"]
LABEL     = "Label"

RANDOM_STATE = 42
TRAIN_FRAC = 0.50
VAL_FRAC   = 0.25
TEST_FRAC  = 0.25

CV_FOLDS = 10
N_JOBS = -1

GROUP_CANDS = ["Flow ID"]

OUT_CV_CSV       = "cv_results.csv"
OUT_VAL_CSV      = "val_results_wide.csv"
OUT_TEST_CSV     = "test_results_wide.csv"

PROFILES = {
    
    "base":   {"packet_loss":0.00, "iat_jitter":0.00, "dur_drift":0.00, "quantize":0.0,  "clip_pctl":None},
    
    "light":  {"packet_loss":0.05, "iat_jitter":0.05, "dur_drift":0.02, "quantize":1.0,  "clip_pctl":99.5},
    
    "medium": {"packet_loss":0.15, "iat_jitter":0.10, "dur_drift":0.05, "quantize":2.0,  "clip_pctl":99.0},
    
    "heavy":  {"packet_loss":0.30, "iat_jitter":0.20, "dur_drift":0.10, "quantize":5.0,  "clip_pctl":98.0},
    
    "extreme":{"packet_loss":0.50, "iat_jitter":0.35, "dur_drift":0.20, "quantize":10.0, "clip_pctl":97.0},
}

def binarize_label(x):
    # 1 = DDoS, 0 = Benign
    return 1 if "ddos" in str(x).lower() else 0

def load_df(path: str) -> pd.DataFrame:
    assert os.path.exists(path), f"No file: {path}"
    head = pd.read_csv(path, nrows=0).columns.tolist()
    miss = [c for c in FEATURES+[LABEL] if c not in head]
    if miss:
        raise ValueError(f"Require Column Missing: {miss}\nFile Column Example: {head[:12]} ...")
    usecols = list(set(FEATURES + [LABEL] + [c for c in GROUP_CANDS if c in head]))
    df = pd.read_csv(path, usecols=usecols).dropna(subset=FEATURES+[LABEL]).copy()
    df[LABEL] = df[LABEL].map(binarize_label).astype(int)
    for f in FEATURES:
        df[f] = pd.to_numeric(df[f], errors="coerce")
    df = df.dropna(subset=FEATURES).reset_index(drop=True)
    return df

def split_50_25_25(df: pd.DataFrame):
    
    gcol = next((c for c in GROUP_CANDS if c in df.columns), None)
    if gcol:
        gss1 = GroupShuffleSplit(n_splits=1, test_size=TEST_FRAC, random_state=RANDOM_STATE)
        idx_trva, idx_te = next(gss1.split(df, groups=df[gcol]))
        tmp, test = df.iloc[idx_trva].reset_index(drop=True), df.iloc[idx_te].reset_index(drop=True)
        gss2 = GroupShuffleSplit(n_splits=1, test_size=VAL_FRAC/(1-TEST_FRAC), random_state=RANDOM_STATE)
        idx_tr, idx_va = next(gss2.split(tmp, groups=tmp[gcol]))
        train, val = tmp.iloc[idx_tr].reset_index(drop=True), tmp.iloc[idx_va].reset_index(drop=True)
    else:
        X, y = df[FEATURES], df[LABEL]
        X_tmp, X_te, y_tmp, y_te = train_test_split(
            X, y, test_size=TEST_FRAC, random_state=RANDOM_STATE, stratify=y
        )
        X_tr, X_va, y_tr, y_va = train_test_split(
            X_tmp, y_tmp, test_size=VAL_FRAC/(1-TEST_FRAC), random_state=RANDOM_STATE, stratify=y_tmp
        )
        train = pd.concat([X_tr, y_tr], axis=1).reset_index(drop=True)
        val   = pd.concat([X_va, y_va], axis=1).reset_index(drop=True)
        test  = pd.concat([X_te, y_te], axis=1).reset_index(drop=True)

    n = len(df)
    print(f"[SPLIT] total={n}, train={len(train)} ({len(train)/n:.2%}), "
          f"val={len(val)} ({len(val)/n:.2%}), test={len(test)} ({len(test)/n:.2%})")
    return train, val, test

def apply_noise(X_df: pd.DataFrame, cfg: dict, seed=42) -> pd.DataFrame:
    
    rng = np.random.default_rng(seed)
    Xn = X_df.copy()

    # (1) Cliping
    if cfg.get("clip_pctl"):
        low = Xn.quantile((100 - cfg["clip_pctl"]) / 100.0)
        high = Xn.quantile(cfg["clip_pctl"] / 100.0)
        Xn = Xn.clip(lower=low, upper=high, axis=1)

    # (2) Packet Loss → Tot Fwd Pkts decay
    if "Tot Fwd Pkts" in Xn.columns and cfg.get("packet_loss", 0) > 0:
        n = np.maximum(0, np.round(Xn["Tot Fwd Pkts"]).astype(int))
        kept = rng.binomial(n=n, p=max(0.0, 1.0 - cfg["packet_loss"]))
        Xn["Tot Fwd Pkts"] = kept.astype(float)

    # (3) IAT Jitter
    if "Flow IAT Mean" in Xn.columns and cfg.get("iat_jitter", 0) > 0:
        jitter = rng.normal(0.0, cfg["iat_jitter"], size=len(Xn))
        Xn["Flow IAT Mean"] = np.maximum(0.0, Xn["Flow IAT Mean"] * (1.0 + jitter))

    # (4) Duration Drift
    if "Flow Duration" in Xn.columns and cfg.get("dur_drift", 0) > 0:
        drift = rng.normal(0.0, cfg["dur_drift"], size=len(Xn))
        Xn["Flow Duration"] = np.maximum(0.0, Xn["Flow Duration"] * (1.0 + drift))

    # (5) quantize
    q = cfg.get("quantize", 0.0)
    if q and q > 0:
        Xn = (Xn / q).round() * q

    return Xn

def metrics_from_pred(y_true, y_pred):
    return dict(
        Accuracy  = accuracy_score(y_true, y_pred),
        Precision = precision_score(y_true, y_pred, zero_division=0),
        Recall    = recall_score(y_true, y_pred, zero_division=0),
        F1        = f1_score(y_true, y_pred, zero_division=0),
    )


scaler = ColumnTransformer([("num", StandardScaler(), FEATURES)], remainder="drop")
smote  = SMOTE(random_state=RANDOM_STATE, k_neighbors=3)

def make_models():
    return {
        "RF": ImbPipeline(steps=[
            ("prep", "passthrough"),
            ("smote", smote),
            ("clf", RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=RANDOM_STATE))
        ]),
        "DT": ImbPipeline(steps=[
            ("prep", "passthrough"),
            ("smote", smote),
            ("clf", DecisionTreeClassifier(random_state=RANDOM_STATE))
        ]),
        "ADA": ImbPipeline(steps=[
            ("prep", "passthrough"),
            ("smote", smote),
            ("clf", AdaBoostClassifier(n_estimators=300, learning_rate=0.5, random_state=RANDOM_STATE))
        ]),
    
        "SVM": ImbPipeline(steps=[
            ("prep", scaler),
            ("smote", smote),
            ("clf", SVC(kernel="rbf", C=1.0, gamma="scale", probability=False, random_state=RANDOM_STATE))
        ]),
        "KNN": ImbPipeline(steps=[
            ("prep", scaler),
            ("smote", smote),
            ("clf", KNeighborsClassifier(n_neighbors=5))
        ]),
    }


scoring = {
    "acc":  "accuracy",
    "prec": make_scorer(precision_score, zero_division=0),
    "rec":  make_scorer(recall_score,    zero_division=0),
    "f1":   make_scorer(f1_score,        zero_division=0),
}


def run():
    
    df = load_df(DATA_PATH)
    train, val, test = split_50_25_25(df)

    
    X_train, y_train = train[FEATURES], train[LABEL]
    X_val,   y_val   = val[FEATURES],   val[LABEL]
    X_test,  y_test  = test[FEATURES],  test[LABEL]

    print("[CLASS DIST] Train:", y_train.value_counts().to_dict(),
          "| Val:", y_val.value_counts().to_dict(),
          "| Test:", y_test.value_counts().to_dict())

   
    models = make_models()

    
    cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    cv_rows = []
    for name, pipe in models.items():
        scores = cross_validate(
            pipe, X_train, y_train,
            cv=cv, scoring=scoring, n_jobs=N_JOBS, return_train_score=False
        )
        cv_rows.append({
            "Model": name,
            "CV_Acc_mean":  float(np.mean(scores["test_acc"])),
            "CV_Acc_std":   float(np.std(scores["test_acc"])),
            "CV_Prec_mean": float(np.mean(scores["test_prec"])),
            "CV_Prec_std":  float(np.std(scores["test_prec"])),
            "CV_Rec_mean":  float(np.mean(scores["test_rec"])),
            "CV_Rec_std":   float(np.std(scores["test_rec"])),
            "CV_F1_mean":   float(np.mean(scores["test_f1"])),
            "CV_F1_std":    float(np.std(scores["test_f1"])),
        })
    cv_df = pd.DataFrame(cv_rows).set_index("Model").sort_index()
    print("\n===== 10-Fold CV on TRAIN (with SMOTE) =====")
    print(cv_df.round(4))
    cv_df.to_csv(OUT_CV_CSV)
    print(f"[Saved] {OUT_CV_CSV}")

    
    def eval_by_profiles(X_base: pd.DataFrame, y_true: pd.Series, fitted_models: dict, seed=RANDOM_STATE):
        rows = []
        for model_name, model in fitted_models.items():
            row = {"Model": model_name}
            for noise_name, cfg in PROFILES.items():
                X_eval = X_base if noise_name == "base" else apply_noise(X_base, cfg, seed=seed)
                y_pred = model.predict(X_eval)
                m = {
                    "Accuracy":  accuracy_score(y_true, y_pred),
                    "Precision": precision_score(y_true, y_pred, zero_division=0),
                    "Recall":    recall_score(y_true, y_pred, zero_division=0),
                    "F1":        f1_score(y_true, y_pred, zero_division=0),
                }
                
                for k, v in m.items():
                    row[f"{k}_{noise_name}"] = v
            rows.append(row)
        wide = pd.DataFrame(rows).set_index("Model").sort_index()
        
        metrics_order = ["Accuracy","Precision","Recall","F1"]
        noise_order = list(PROFILES.keys())
        wide = wide.reindex(columns=[f"{m}_{n}" for m in metrics_order for n in noise_order])
        return wide

    
    fitted = {}
    for name, pipe in models.items():
        pipe.fit(X_train, y_train)  
        fitted[name] = pipe

    
    val_wide = eval_by_profiles(X_val, y_val, fitted)
    print("\n===== Validation (per noise profile) =====")
    print(val_wide.round(4))
    val_wide.to_csv(OUT_VAL_CSV)
    print(f"[Saved] {OUT_VAL_CSV}")

    
    test_wide = eval_by_profiles(X_test, y_test, fitted)
    print("\n===== Test (per noise profile) =====")
    print(test_wide.round(4))
    test_wide.to_csv(OUT_TEST_CSV)
    print(f"[Saved] {OUT_TEST_CSV}")

if __name__ == "__main__":
    run()


[SPLIT] total=3940, train=2027 (51.45%), val=955 (24.24%), test=958 (24.31%)
[CLASS DIST] Train: {0: 1102, 1: 925} | Val: {0: 492, 1: 463} | Test: {0: 490, 1: 468}

===== 10-Fold CV on TRAIN (with SMOTE) =====
       CV_Acc_mean  CV_Acc_std  CV_Prec_mean  CV_Prec_std  CV_Rec_mean  \
Model                                                                    
ADA         0.9901      0.0070        0.9831       0.0133       0.9957   
DT          0.9877      0.0080        0.9860       0.0107       0.9870   
KNN         0.9931      0.0059        0.9883       0.0088       0.9967   
RF          0.9906      0.0071        0.9851       0.0117       0.9946   
SVM         0.9926      0.0055        0.9842       0.0117       1.0000   

       CV_Rec_std  CV_F1_mean  CV_F1_std  
Model                                     
ADA        0.0072      0.9893     0.0075  
DT         0.0081      0.9865     0.0088  
KNN        0.0098      0.9924     0.0065  
RF         0.0073      0.9898     0.0077  
SVM        0.

In [2]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GroupShuffleSplit, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, make_scorer
)
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

DATA_PATH = "Merged_final_IoUT_shallow.csv" 
FEATURES  = ["Flow Duration", "Flow IAT Mean", "Tot Fwd Pkts"]
LABEL     = "Label"

RANDOM_STATE = 42
TRAIN_FRAC = 0.50
VAL_FRAC   = 0.25
TEST_FRAC  = 0.25

CV_FOLDS = 10
N_JOBS = -1

GROUP_CANDS = ["Flow ID"]

OUT_CV_CSV       = "cv_results.csv"
OUT_VAL_CSV      = "val_results_wide.csv"
OUT_TEST_CSV     = "test_results_wide.csv"

PROFILES = {
    
    "base":   {"packet_loss":0.00, "iat_jitter":0.00, "dur_drift":0.00, "quantize":0.0,  "clip_pctl":None},
    
    "light":  {"packet_loss":0.05, "iat_jitter":0.05, "dur_drift":0.02, "quantize":1.0,  "clip_pctl":99.5},
    
    "medium": {"packet_loss":0.15, "iat_jitter":0.10, "dur_drift":0.05, "quantize":2.0,  "clip_pctl":99.0},
    
    "heavy":  {"packet_loss":0.30, "iat_jitter":0.20, "dur_drift":0.10, "quantize":5.0,  "clip_pctl":98.0},
    
    "extreme":{"packet_loss":0.50, "iat_jitter":0.35, "dur_drift":0.20, "quantize":10.0, "clip_pctl":97.0},
}

def binarize_label(x):
    # 1 = DDoS, 0 = Benign
    return 1 if "ddos" in str(x).lower() else 0

def load_df(path: str) -> pd.DataFrame:
    assert os.path.exists(path), f"No file: {path}"
    head = pd.read_csv(path, nrows=0).columns.tolist()
    miss = [c for c in FEATURES+[LABEL] if c not in head]
    if miss:
        raise ValueError(f"Require Column Missing: {miss}\nFile Column Example: {head[:12]} ...")
    usecols = list(set(FEATURES + [LABEL] + [c for c in GROUP_CANDS if c in head]))
    df = pd.read_csv(path, usecols=usecols).dropna(subset=FEATURES+[LABEL]).copy()
    df[LABEL] = df[LABEL].map(binarize_label).astype(int)
    for f in FEATURES:
        df[f] = pd.to_numeric(df[f], errors="coerce")
    df = df.dropna(subset=FEATURES).reset_index(drop=True)
    return df

def split_50_25_25(df: pd.DataFrame):
    
    gcol = next((c for c in GROUP_CANDS if c in df.columns), None)
    if gcol:
        gss1 = GroupShuffleSplit(n_splits=1, test_size=TEST_FRAC, random_state=RANDOM_STATE)
        idx_trva, idx_te = next(gss1.split(df, groups=df[gcol]))
        tmp, test = df.iloc[idx_trva].reset_index(drop=True), df.iloc[idx_te].reset_index(drop=True)
        gss2 = GroupShuffleSplit(n_splits=1, test_size=VAL_FRAC/(1-TEST_FRAC), random_state=RANDOM_STATE)
        idx_tr, idx_va = next(gss2.split(tmp, groups=tmp[gcol]))
        train, val = tmp.iloc[idx_tr].reset_index(drop=True), tmp.iloc[idx_va].reset_index(drop=True)
    else:
        X, y = df[FEATURES], df[LABEL]
        X_tmp, X_te, y_tmp, y_te = train_test_split(
            X, y, test_size=TEST_FRAC, random_state=RANDOM_STATE, stratify=y
        )
        X_tr, X_va, y_tr, y_va = train_test_split(
            X_tmp, y_tmp, test_size=VAL_FRAC/(1-TEST_FRAC), random_state=RANDOM_STATE, stratify=y_tmp
        )
        train = pd.concat([X_tr, y_tr], axis=1).reset_index(drop=True)
        val   = pd.concat([X_va, y_va], axis=1).reset_index(drop=True)
        test  = pd.concat([X_te, y_te], axis=1).reset_index(drop=True)

    n = len(df)
    print(f"[SPLIT] total={n}, train={len(train)} ({len(train)/n:.2%}), "
          f"val={len(val)} ({len(val)/n:.2%}), test={len(test)} ({len(test)/n:.2%})")
    return train, val, test

def apply_noise(X_df: pd.DataFrame, cfg: dict, seed=42) -> pd.DataFrame:
    
    rng = np.random.default_rng(seed)
    Xn = X_df.copy()

    # (1) Cliping
    if cfg.get("clip_pctl"):
        low = Xn.quantile((100 - cfg["clip_pctl"]) / 100.0)
        high = Xn.quantile(cfg["clip_pctl"] / 100.0)
        Xn = Xn.clip(lower=low, upper=high, axis=1)

    # (2) Packet Loss → Tot Fwd Pkts decay
    if "Tot Fwd Pkts" in Xn.columns and cfg.get("packet_loss", 0) > 0:
        n = np.maximum(0, np.round(Xn["Tot Fwd Pkts"]).astype(int))
        kept = rng.binomial(n=n, p=max(0.0, 1.0 - cfg["packet_loss"]))
        Xn["Tot Fwd Pkts"] = kept.astype(float)

    # (3) IAT Jitter
    if "Flow IAT Mean" in Xn.columns and cfg.get("iat_jitter", 0) > 0:
        jitter = rng.normal(0.0, cfg["iat_jitter"], size=len(Xn))
        Xn["Flow IAT Mean"] = np.maximum(0.0, Xn["Flow IAT Mean"] * (1.0 + jitter))

    # (4) Duration Drift
    if "Flow Duration" in Xn.columns and cfg.get("dur_drift", 0) > 0:
        drift = rng.normal(0.0, cfg["dur_drift"], size=len(Xn))
        Xn["Flow Duration"] = np.maximum(0.0, Xn["Flow Duration"] * (1.0 + drift))

    # (5) quantize
    q = cfg.get("quantize", 0.0)
    if q and q > 0:
        Xn = (Xn / q).round() * q

    return Xn

def metrics_from_pred(y_true, y_pred):
    return dict(
        Accuracy  = accuracy_score(y_true, y_pred),
        Precision = precision_score(y_true, y_pred, zero_division=0),
        Recall    = recall_score(y_true, y_pred, zero_division=0),
        F1        = f1_score(y_true, y_pred, zero_division=0),
    )


scaler = ColumnTransformer([("num", StandardScaler(), FEATURES)], remainder="drop")
smote  = SMOTE(random_state=RANDOM_STATE, k_neighbors=3)

def make_models():
    return {
        "RF": ImbPipeline(steps=[
            ("prep", "passthrough"),
            ("smote", smote),
            ("clf", RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=RANDOM_STATE))
        ]),
        "DT": ImbPipeline(steps=[
            ("prep", "passthrough"),
            ("smote", smote),
            ("clf", DecisionTreeClassifier(random_state=RANDOM_STATE))
        ]),
        "ADA": ImbPipeline(steps=[
            ("prep", "passthrough"),
            ("smote", smote),
            ("clf", AdaBoostClassifier(n_estimators=300, learning_rate=0.5, random_state=RANDOM_STATE))
        ]),
    
        "SVM": ImbPipeline(steps=[
            ("prep", scaler),
            ("smote", smote),
            ("clf", SVC(kernel="rbf", C=1.0, gamma="scale", probability=False, random_state=RANDOM_STATE))
        ]),
        "KNN": ImbPipeline(steps=[
            ("prep", scaler),
            ("smote", smote),
            ("clf", KNeighborsClassifier(n_neighbors=5))
        ]),
    }


scoring = {
    "acc":  "accuracy",
    "prec": make_scorer(precision_score, zero_division=0),
    "rec":  make_scorer(recall_score,    zero_division=0),
    "f1":   make_scorer(f1_score,        zero_division=0),
}


def run():
    
    df = load_df(DATA_PATH)
    train, val, test = split_50_25_25(df)

    
    X_train, y_train = train[FEATURES], train[LABEL]
    X_val,   y_val   = val[FEATURES],   val[LABEL]
    X_test,  y_test  = test[FEATURES],  test[LABEL]

    print("[CLASS DIST] Train:", y_train.value_counts().to_dict(),
          "| Val:", y_val.value_counts().to_dict(),
          "| Test:", y_test.value_counts().to_dict())

   
    models = make_models()

    
    cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    cv_rows = []
    for name, pipe in models.items():
        scores = cross_validate(
            pipe, X_train, y_train,
            cv=cv, scoring=scoring, n_jobs=N_JOBS, return_train_score=False
        )
        cv_rows.append({
            "Model": name,
            "CV_Acc_mean":  float(np.mean(scores["test_acc"])),
            "CV_Acc_std":   float(np.std(scores["test_acc"])),
            "CV_Prec_mean": float(np.mean(scores["test_prec"])),
            "CV_Prec_std":  float(np.std(scores["test_prec"])),
            "CV_Rec_mean":  float(np.mean(scores["test_rec"])),
            "CV_Rec_std":   float(np.std(scores["test_rec"])),
            "CV_F1_mean":   float(np.mean(scores["test_f1"])),
            "CV_F1_std":    float(np.std(scores["test_f1"])),
        })
    cv_df = pd.DataFrame(cv_rows).set_index("Model").sort_index()
    print("\n===== 10-Fold CV on TRAIN (with SMOTE) =====")
    print(cv_df.round(4))
    cv_df.to_csv(OUT_CV_CSV)
    print(f"[Saved] {OUT_CV_CSV}")

    
    def eval_by_profiles(X_base: pd.DataFrame, y_true: pd.Series, fitted_models: dict, seed=RANDOM_STATE):
        rows = []
        for model_name, model in fitted_models.items():
            row = {"Model": model_name}
            for noise_name, cfg in PROFILES.items():
                X_eval = X_base if noise_name == "base" else apply_noise(X_base, cfg, seed=seed)
                y_pred = model.predict(X_eval)
                m = {
                    "Accuracy":  accuracy_score(y_true, y_pred),
                    "Precision": precision_score(y_true, y_pred, zero_division=0),
                    "Recall":    recall_score(y_true, y_pred, zero_division=0),
                    "F1":        f1_score(y_true, y_pred, zero_division=0),
                }
                
                for k, v in m.items():
                    row[f"{k}_{noise_name}"] = v
            rows.append(row)
        wide = pd.DataFrame(rows).set_index("Model").sort_index()
        
        metrics_order = ["Accuracy","Precision","Recall","F1"]
        noise_order = list(PROFILES.keys())
        wide = wide.reindex(columns=[f"{m}_{n}" for m in metrics_order for n in noise_order])
        return wide

    
    fitted = {}
    for name, pipe in models.items():
        pipe.fit(X_train, y_train)  
        fitted[name] = pipe

    
    val_wide = eval_by_profiles(X_val, y_val, fitted)
    print("\n===== Validation (per noise profile) =====")
    print(val_wide.round(4))
    val_wide.to_csv(OUT_VAL_CSV)
    print(f"[Saved] {OUT_VAL_CSV}")

    
    test_wide = eval_by_profiles(X_test, y_test, fitted)
    print("\n===== Test (per noise profile) =====")
    print(test_wide.round(4))
    test_wide.to_csv(OUT_TEST_CSV)
    print(f"[Saved] {OUT_TEST_CSV}")

if __name__ == "__main__":
    run()


[SPLIT] total=4233, train=2038 (48.15%), val=1034 (24.43%), test=1161 (27.43%)
[CLASS DIST] Train: {0: 1023, 1: 1015} | Val: {0: 526, 1: 508} | Test: {0: 654, 1: 507}

===== 10-Fold CV on TRAIN (with SMOTE) =====
       CV_Acc_mean  CV_Acc_std  CV_Prec_mean  CV_Prec_std  CV_Rec_mean  \
Model                                                                    
ADA         0.9975      0.0040        0.9980       0.0039       0.9970   
DT          0.9980      0.0024        0.9971       0.0045       0.9990   
KNN         0.9956      0.0051        0.9961       0.0065       0.9951   
RF          0.9985      0.0023        0.9980       0.0039       0.9990   
SVM         0.9966      0.0044        0.9961       0.0065       0.9971   

       CV_Rec_std  CV_F1_mean  CV_F1_std  
Model                                     
ADA        0.0063      0.9975     0.0040  
DT         0.0029      0.9980     0.0024  
KNN        0.0066      0.9956     0.0051  
RF         0.0029      0.9985     0.0023  
SVM       

In [ ]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GroupShuffleSplit, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, make_scorer
)
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

DATA_PATH = "Merged_final_IoUT_70cm.csv" 
FEATURES  = ["Flow Duration", "Flow IAT Mean", "Tot Fwd Pkts"]
LABEL     = "Label"

RANDOM_STATE = 42
TRAIN_FRAC = 0.50
VAL_FRAC   = 0.25
TEST_FRAC  = 0.25

CV_FOLDS = 10
N_JOBS = -1

GROUP_CANDS = ["Flow ID"]

OUT_CV_CSV       = "cv_results.csv"
OUT_VAL_CSV      = "val_results_wide.csv"
OUT_TEST_CSV     = "test_results_wide.csv"

PROFILES = {
    
    "base":   {"packet_loss":0.00, "iat_jitter":0.00, "dur_drift":0.00, "quantize":0.0,  "clip_pctl":None},
    
    "light":  {"packet_loss":0.05, "iat_jitter":0.05, "dur_drift":0.02, "quantize":1.0,  "clip_pctl":99.5},
    
    "medium": {"packet_loss":0.15, "iat_jitter":0.10, "dur_drift":0.05, "quantize":2.0,  "clip_pctl":99.0},
    
    "heavy":  {"packet_loss":0.30, "iat_jitter":0.20, "dur_drift":0.10, "quantize":5.0,  "clip_pctl":98.0},
    
    "extreme":{"packet_loss":0.50, "iat_jitter":0.35, "dur_drift":0.20, "quantize":10.0, "clip_pctl":97.0},
}

def binarize_label(x):
    # 1 = DDoS, 0 = Benign
    return 1 if "ddos" in str(x).lower() else 0

def load_df(path: str) -> pd.DataFrame:
    assert os.path.exists(path), f"No file: {path}"
    head = pd.read_csv(path, nrows=0).columns.tolist()
    miss = [c for c in FEATURES+[LABEL] if c not in head]
    if miss:
        raise ValueError(f"Require Column Missing: {miss}\nFile Column Example: {head[:12]} ...")
    usecols = list(set(FEATURES + [LABEL] + [c for c in GROUP_CANDS if c in head]))
    df = pd.read_csv(path, usecols=usecols).dropna(subset=FEATURES+[LABEL]).copy()
    df[LABEL] = df[LABEL].map(binarize_label).astype(int)
    for f in FEATURES:
        df[f] = pd.to_numeric(df[f], errors="coerce")
    df = df.dropna(subset=FEATURES).reset_index(drop=True)
    return df

def split_50_25_25(df: pd.DataFrame):
    
    gcol = next((c for c in GROUP_CANDS if c in df.columns), None)
    if gcol:
        gss1 = GroupShuffleSplit(n_splits=1, test_size=TEST_FRAC, random_state=RANDOM_STATE)
        idx_trva, idx_te = next(gss1.split(df, groups=df[gcol]))
        tmp, test = df.iloc[idx_trva].reset_index(drop=True), df.iloc[idx_te].reset_index(drop=True)
        gss2 = GroupShuffleSplit(n_splits=1, test_size=VAL_FRAC/(1-TEST_FRAC), random_state=RANDOM_STATE)
        idx_tr, idx_va = next(gss2.split(tmp, groups=tmp[gcol]))
        train, val = tmp.iloc[idx_tr].reset_index(drop=True), tmp.iloc[idx_va].reset_index(drop=True)
    else:
        X, y = df[FEATURES], df[LABEL]
        X_tmp, X_te, y_tmp, y_te = train_test_split(
            X, y, test_size=TEST_FRAC, random_state=RANDOM_STATE, stratify=y
        )
        X_tr, X_va, y_tr, y_va = train_test_split(
            X_tmp, y_tmp, test_size=VAL_FRAC/(1-TEST_FRAC), random_state=RANDOM_STATE, stratify=y_tmp
        )
        train = pd.concat([X_tr, y_tr], axis=1).reset_index(drop=True)
        val   = pd.concat([X_va, y_va], axis=1).reset_index(drop=True)
        test  = pd.concat([X_te, y_te], axis=1).reset_index(drop=True)

    n = len(df)
    print(f"[SPLIT] total={n}, train={len(train)} ({len(train)/n:.2%}), "
          f"val={len(val)} ({len(val)/n:.2%}), test={len(test)} ({len(test)/n:.2%})")
    return train, val, test

def apply_noise(X_df: pd.DataFrame, cfg: dict, seed=42) -> pd.DataFrame:
    
    rng = np.random.default_rng(seed)
    Xn = X_df.copy()

    # (1) Cliping
    if cfg.get("clip_pctl"):
        low = Xn.quantile((100 - cfg["clip_pctl"]) / 100.0)
        high = Xn.quantile(cfg["clip_pctl"] / 100.0)
        Xn = Xn.clip(lower=low, upper=high, axis=1)

    # (2) Packet Loss → Tot Fwd Pkts decay
    if "Tot Fwd Pkts" in Xn.columns and cfg.get("packet_loss", 0) > 0:
        n = np.maximum(0, np.round(Xn["Tot Fwd Pkts"]).astype(int))
        kept = rng.binomial(n=n, p=max(0.0, 1.0 - cfg["packet_loss"]))
        Xn["Tot Fwd Pkts"] = kept.astype(float)

    # (3) IAT Jitter
    if "Flow IAT Mean" in Xn.columns and cfg.get("iat_jitter", 0) > 0:
        jitter = rng.normal(0.0, cfg["iat_jitter"], size=len(Xn))
        Xn["Flow IAT Mean"] = np.maximum(0.0, Xn["Flow IAT Mean"] * (1.0 + jitter))

    # (4) Duration Drift
    if "Flow Duration" in Xn.columns and cfg.get("dur_drift", 0) > 0:
        drift = rng.normal(0.0, cfg["dur_drift"], size=len(Xn))
        Xn["Flow Duration"] = np.maximum(0.0, Xn["Flow Duration"] * (1.0 + drift))

    # (5) quantize
    q = cfg.get("quantize", 0.0)
    if q and q > 0:
        Xn = (Xn / q).round() * q

    return Xn

def metrics_from_pred(y_true, y_pred):
    return dict(
        Accuracy  = accuracy_score(y_true, y_pred),
        Precision = precision_score(y_true, y_pred, zero_division=0),
        Recall    = recall_score(y_true, y_pred, zero_division=0),
        F1        = f1_score(y_true, y_pred, zero_division=0),
    )


scaler = ColumnTransformer([("num", StandardScaler(), FEATURES)], remainder="drop")
smote  = SMOTE(random_state=RANDOM_STATE, k_neighbors=3)

def make_models():
    return {
        "RF": ImbPipeline(steps=[
            ("prep", "passthrough"),
            ("smote", smote),
            ("clf", RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=RANDOM_STATE))
        ]),
        "DT": ImbPipeline(steps=[
            ("prep", "passthrough"),
            ("smote", smote),
            ("clf", DecisionTreeClassifier(random_state=RANDOM_STATE))
        ]),
        "ADA": ImbPipeline(steps=[
            ("prep", "passthrough"),
            ("smote", smote),
            ("clf", AdaBoostClassifier(n_estimators=300, learning_rate=0.5, random_state=RANDOM_STATE))
        ]),
    
        "SVM": ImbPipeline(steps=[
            ("prep", scaler),
            ("smote", smote),
            ("clf", SVC(kernel="rbf", C=1.0, gamma="scale", probability=False, random_state=RANDOM_STATE))
        ]),
        "KNN": ImbPipeline(steps=[
            ("prep", scaler),
            ("smote", smote),
            ("clf", KNeighborsClassifier(n_neighbors=5))
        ]),
    }


scoring = {
    "acc":  "accuracy",
    "prec": make_scorer(precision_score, zero_division=0),
    "rec":  make_scorer(recall_score,    zero_division=0),
    "f1":   make_scorer(f1_score,        zero_division=0),
}


def run():
    
    df = load_df(DATA_PATH)
    train, val, test = split_50_25_25(df)

    
    X_train, y_train = train[FEATURES], train[LABEL]
    X_val,   y_val   = val[FEATURES],   val[LABEL]
    X_test,  y_test  = test[FEATURES],  test[LABEL]

    print("[CLASS DIST] Train:", y_train.value_counts().to_dict(),
          "| Val:", y_val.value_counts().to_dict(),
          "| Test:", y_test.value_counts().to_dict())

   
    models = make_models()

    
    cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    cv_rows = []
    for name, pipe in models.items():
        scores = cross_validate(
            pipe, X_train, y_train,
            cv=cv, scoring=scoring, n_jobs=N_JOBS, return_train_score=False
        )
        cv_rows.append({
            "Model": name,
            "CV_Acc_mean":  float(np.mean(scores["test_acc"])),
            "CV_Acc_std":   float(np.std(scores["test_acc"])),
            "CV_Prec_mean": float(np.mean(scores["test_prec"])),
            "CV_Prec_std":  float(np.std(scores["test_prec"])),
            "CV_Rec_mean":  float(np.mean(scores["test_rec"])),
            "CV_Rec_std":   float(np.std(scores["test_rec"])),
            "CV_F1_mean":   float(np.mean(scores["test_f1"])),
            "CV_F1_std":    float(np.std(scores["test_f1"])),
        })
    cv_df = pd.DataFrame(cv_rows).set_index("Model").sort_index()
    print("\n===== 10-Fold CV on TRAIN (with SMOTE) =====")
    print(cv_df.round(4))
    cv_df.to_csv(OUT_CV_CSV)
    print(f"[Saved] {OUT_CV_CSV}")

    
    def eval_by_profiles(X_base: pd.DataFrame, y_true: pd.Series, fitted_models: dict, seed=RANDOM_STATE):
        rows = []
        for model_name, model in fitted_models.items():
            row = {"Model": model_name}
            for noise_name, cfg in PROFILES.items():
                X_eval = X_base if noise_name == "base" else apply_noise(X_base, cfg, seed=seed)
                y_pred = model.predict(X_eval)
                m = {
                    "Accuracy":  accuracy_score(y_true, y_pred),
                    "Precision": precision_score(y_true, y_pred, zero_division=0),
                    "Recall":    recall_score(y_true, y_pred, zero_division=0),
                    "F1":        f1_score(y_true, y_pred, zero_division=0),
                }
                
                for k, v in m.items():
                    row[f"{k}_{noise_name}"] = v
            rows.append(row)
        wide = pd.DataFrame(rows).set_index("Model").sort_index()
        
        metrics_order = ["Accuracy","Precision","Recall","F1"]
        noise_order = list(PROFILES.keys())
        wide = wide.reindex(columns=[f"{m}_{n}" for m in metrics_order for n in noise_order])
        return wide

    
    fitted = {}
    for name, pipe in models.items():
        pipe.fit(X_train, y_train)  
        fitted[name] = pipe

    
    val_wide = eval_by_profiles(X_val, y_val, fitted)
    print("\n===== Validation (per noise profile) =====")
    print(val_wide.round(4))
    val_wide.to_csv(OUT_VAL_CSV)
    print(f"[Saved] {OUT_VAL_CSV}")

    
    test_wide = eval_by_profiles(X_test, y_test, fitted)
    print("\n===== Test (per noise profile) =====")
    print(test_wide.round(4))
    test_wide.to_csv(OUT_TEST_CSV)
    print(f"[Saved] {OUT_TEST_CSV}")

if __name__ == "__main__":
    run()


[SPLIT] total=1591, train=887 (55.75%), val=340 (21.37%), test=364 (22.88%)
[CLASS DIST] Train: {0: 669, 1: 218} | Val: {0: 227, 1: 113} | Test: {0: 249, 1: 115}

===== 10-Fold CV on TRAIN (with SMOTE) =====
       CV_Acc_mean  CV_Acc_std  CV_Prec_mean  CV_Prec_std  CV_Rec_mean  CV_Rec_std  CV_F1_mean  CV_F1_std
Model                                                                                                    
ADA         0.9245      0.0348        0.8179       0.0823       0.8989      0.0450      0.8556     0.0631
DT          0.9358      0.0256        0.8608       0.0349       0.8812      0.0915      0.8690     0.0557
KNN         0.9256      0.0242        0.8196       0.0583       0.8991      0.0529      0.8562     0.0446
RF          0.9347      0.0234        0.8591       0.0448       0.8812      0.0845      0.8676     0.0500
SVM         0.9403      0.0322        0.9075       0.0846       0.8489      0.0578      0.8758     0.0630
[Saved] cv_results.csv

===== Validation (per nois

In [3]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GroupShuffleSplit, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, make_scorer
)
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

DATA_PATH = "real_world_ddos_1m.csv"
FEATURES  = ["Flow Duration", "Flow IAT Mean", "Tot Fwd Pkts"]
LABEL     = "Label"

RANDOM_STATE = 42
TRAIN_FRAC = 0.50
VAL_FRAC   = 0.25
TEST_FRAC  = 0.25

CV_FOLDS = 10
N_JOBS = -1

GROUP_CANDS = ["Flow ID"]

OUT_CV_CSV       = "cv_results.csv"
OUT_VAL_CSV      = "val_results_wide.csv"
OUT_TEST_CSV     = "test_results_wide.csv"

PROFILES = {
    
    "base":   {"packet_loss":0.00, "iat_jitter":0.00, "dur_drift":0.00, "quantize":0.0,  "clip_pctl":None},
    
    "light":  {"packet_loss":0.05, "iat_jitter":0.05, "dur_drift":0.02, "quantize":1.0,  "clip_pctl":99.5},
    
    "medium": {"packet_loss":0.15, "iat_jitter":0.10, "dur_drift":0.05, "quantize":2.0,  "clip_pctl":99.0},
    
    "heavy":  {"packet_loss":0.30, "iat_jitter":0.20, "dur_drift":0.10, "quantize":5.0,  "clip_pctl":98.0},
    
    "extreme":{"packet_loss":0.50, "iat_jitter":0.35, "dur_drift":0.20, "quantize":10.0, "clip_pctl":97.0},
}

def binarize_label(x):
    # 1 = DDoS, 0 = Benign
    return 1 if "ddos" in str(x).lower() else 0

def load_df(path: str) -> pd.DataFrame:
    assert os.path.exists(path), f"No file: {path}"
    head = pd.read_csv(path, nrows=0).columns.tolist()
    miss = [c for c in FEATURES+[LABEL] if c not in head]
    if miss:
        raise ValueError(f"Require Column Missing: {miss}\nFile Column Example: {head[:12]} ...")
    usecols = list(set(FEATURES + [LABEL] + [c for c in GROUP_CANDS if c in head]))
    df = pd.read_csv(path, usecols=usecols).dropna(subset=FEATURES+[LABEL]).copy()
    df[LABEL] = df[LABEL].map(binarize_label).astype(int)
    for f in FEATURES:
        df[f] = pd.to_numeric(df[f], errors="coerce")
    df = df.dropna(subset=FEATURES).reset_index(drop=True)
    return df

def split_50_25_25(df: pd.DataFrame):
    
    gcol = next((c for c in GROUP_CANDS if c in df.columns), None)
    if gcol:
        gss1 = GroupShuffleSplit(n_splits=1, test_size=TEST_FRAC, random_state=RANDOM_STATE)
        idx_trva, idx_te = next(gss1.split(df, groups=df[gcol]))
        tmp, test = df.iloc[idx_trva].reset_index(drop=True), df.iloc[idx_te].reset_index(drop=True)
        gss2 = GroupShuffleSplit(n_splits=1, test_size=VAL_FRAC/(1-TEST_FRAC), random_state=RANDOM_STATE)
        idx_tr, idx_va = next(gss2.split(tmp, groups=tmp[gcol]))
        train, val = tmp.iloc[idx_tr].reset_index(drop=True), tmp.iloc[idx_va].reset_index(drop=True)
    else:
        X, y = df[FEATURES], df[LABEL]
        X_tmp, X_te, y_tmp, y_te = train_test_split(
            X, y, test_size=TEST_FRAC, random_state=RANDOM_STATE, stratify=y
        )
        X_tr, X_va, y_tr, y_va = train_test_split(
            X_tmp, y_tmp, test_size=VAL_FRAC/(1-TEST_FRAC), random_state=RANDOM_STATE, stratify=y_tmp
        )
        train = pd.concat([X_tr, y_tr], axis=1).reset_index(drop=True)
        val   = pd.concat([X_va, y_va], axis=1).reset_index(drop=True)
        test  = pd.concat([X_te, y_te], axis=1).reset_index(drop=True)

    n = len(df)
    print(f"[SPLIT] total={n}, train={len(train)} ({len(train)/n:.2%}), "
          f"val={len(val)} ({len(val)/n:.2%}), test={len(test)} ({len(test)/n:.2%})")
    return train, val, test

def apply_noise(X_df: pd.DataFrame, cfg: dict, seed=42) -> pd.DataFrame:
    
    rng = np.random.default_rng(seed)
    Xn = X_df.copy()

    # (1) Cliping
    if cfg.get("clip_pctl"):
        low = Xn.quantile((100 - cfg["clip_pctl"]) / 100.0)
        high = Xn.quantile(cfg["clip_pctl"] / 100.0)
        Xn = Xn.clip(lower=low, upper=high, axis=1)

    # (2) Packet Loss → Tot Fwd Pkts decay
    if "Tot Fwd Pkts" in Xn.columns and cfg.get("packet_loss", 0) > 0:
        n = np.maximum(0, np.round(Xn["Tot Fwd Pkts"]).astype(int))
        kept = rng.binomial(n=n, p=max(0.0, 1.0 - cfg["packet_loss"]))
        Xn["Tot Fwd Pkts"] = kept.astype(float)

    # (3) IAT Jitter
    if "Flow IAT Mean" in Xn.columns and cfg.get("iat_jitter", 0) > 0:
        jitter = rng.normal(0.0, cfg["iat_jitter"], size=len(Xn))
        Xn["Flow IAT Mean"] = np.maximum(0.0, Xn["Flow IAT Mean"] * (1.0 + jitter))

    # (4) Duration Drift
    if "Flow Duration" in Xn.columns and cfg.get("dur_drift", 0) > 0:
        drift = rng.normal(0.0, cfg["dur_drift"], size=len(Xn))
        Xn["Flow Duration"] = np.maximum(0.0, Xn["Flow Duration"] * (1.0 + drift))

    # (5) quantize
    q = cfg.get("quantize", 0.0)
    if q and q > 0:
        Xn = (Xn / q).round() * q

    return Xn

def metrics_from_pred(y_true, y_pred):
    return dict(
        Accuracy  = accuracy_score(y_true, y_pred),
        Precision = precision_score(y_true, y_pred, zero_division=0),
        Recall    = recall_score(y_true, y_pred, zero_division=0),
        F1        = f1_score(y_true, y_pred, zero_division=0),
    )


scaler = ColumnTransformer([("num", StandardScaler(), FEATURES)], remainder="drop")
smote  = SMOTE(random_state=RANDOM_STATE, k_neighbors=3)

def make_models():
    return {
        "RF": ImbPipeline(steps=[
            ("prep", "passthrough"),
            ("smote", smote),
            ("clf", RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=RANDOM_STATE))
        ]),
        "DT": ImbPipeline(steps=[
            ("prep", "passthrough"),
            ("smote", smote),
            ("clf", DecisionTreeClassifier(random_state=RANDOM_STATE))
        ]),
        "ADA": ImbPipeline(steps=[
            ("prep", "passthrough"),
            ("smote", smote),
            ("clf", AdaBoostClassifier(n_estimators=300, learning_rate=0.5, random_state=RANDOM_STATE))
        ]),
    
        "SVM": ImbPipeline(steps=[
            ("prep", scaler),
            ("smote", smote),
            ("clf", SVC(kernel="rbf", C=1.0, gamma="scale", probability=False, random_state=RANDOM_STATE))
        ]),
        "KNN": ImbPipeline(steps=[
            ("prep", scaler),
            ("smote", smote),
            ("clf", KNeighborsClassifier(n_neighbors=5))
        ]),
    }


scoring = {
    "acc":  "accuracy",
    "prec": make_scorer(precision_score, zero_division=0),
    "rec":  make_scorer(recall_score,    zero_division=0),
    "f1":   make_scorer(f1_score,        zero_division=0),
}


def run():
    
    df = load_df(DATA_PATH)
    train, val, test = split_50_25_25(df)

    
    X_train, y_train = train[FEATURES], train[LABEL]
    X_val,   y_val   = val[FEATURES],   val[LABEL]
    X_test,  y_test  = test[FEATURES],  test[LABEL]

    print("[CLASS DIST] Train:", y_train.value_counts().to_dict(),
          "| Val:", y_val.value_counts().to_dict(),
          "| Test:", y_test.value_counts().to_dict())

   
    models = make_models()

    
    cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    cv_rows = []
    for name, pipe in models.items():
        scores = cross_validate(
            pipe, X_train, y_train,
            cv=cv, scoring=scoring, n_jobs=N_JOBS, return_train_score=False
        )
        cv_rows.append({
            "Model": name,
            "CV_Acc_mean":  float(np.mean(scores["test_acc"])),
            "CV_Acc_std":   float(np.std(scores["test_acc"])),
            "CV_Prec_mean": float(np.mean(scores["test_prec"])),
            "CV_Prec_std":  float(np.std(scores["test_prec"])),
            "CV_Rec_mean":  float(np.mean(scores["test_rec"])),
            "CV_Rec_std":   float(np.std(scores["test_rec"])),
            "CV_F1_mean":   float(np.mean(scores["test_f1"])),
            "CV_F1_std":    float(np.std(scores["test_f1"])),
        })
    cv_df = pd.DataFrame(cv_rows).set_index("Model").sort_index()
    print("\n===== 10-Fold CV on TRAIN (with SMOTE) =====")
    print(cv_df.round(4))
    cv_df.to_csv(OUT_CV_CSV)
    print(f"[Saved] {OUT_CV_CSV}")

    
    def eval_by_profiles(X_base: pd.DataFrame, y_true: pd.Series, fitted_models: dict, seed=RANDOM_STATE):
        rows = []
        for model_name, model in fitted_models.items():
            row = {"Model": model_name}
            for noise_name, cfg in PROFILES.items():
                X_eval = X_base if noise_name == "base" else apply_noise(X_base, cfg, seed=seed)
                y_pred = model.predict(X_eval)
                m = {
                    "Accuracy":  accuracy_score(y_true, y_pred),
                    "Precision": precision_score(y_true, y_pred, zero_division=0),
                    "Recall":    recall_score(y_true, y_pred, zero_division=0),
                    "F1":        f1_score(y_true, y_pred, zero_division=0),
                }
                
                for k, v in m.items():
                    row[f"{k}_{noise_name}"] = v
            rows.append(row)
        wide = pd.DataFrame(rows).set_index("Model").sort_index()
        
        metrics_order = ["Accuracy","Precision","Recall","F1"]
        noise_order = list(PROFILES.keys())
        wide = wide.reindex(columns=[f"{m}_{n}" for m in metrics_order for n in noise_order])
        return wide

    
    fitted = {}
    for name, pipe in models.items():
        pipe.fit(X_train, y_train)  
        fitted[name] = pipe

    
    val_wide = eval_by_profiles(X_val, y_val, fitted)
    print("\n===== Validation (per noise profile) =====")
    print(val_wide.round(4))
    val_wide.to_csv(OUT_VAL_CSV)
    print(f"[Saved] {OUT_VAL_CSV}")

    
    test_wide = eval_by_profiles(X_test, y_test, fitted)
    print("\n===== Test (per noise profile) =====")
    print(test_wide.round(4))
    test_wide.to_csv(OUT_TEST_CSV)
    print(f"[Saved] {OUT_TEST_CSV}")

if __name__ == "__main__":
    run()


[SPLIT] total=1864, train=934 (50.11%), val=464 (24.89%), test=466 (25.00%)
[CLASS DIST] Train: {0: 486, 1: 448} | Val: {0: 241, 1: 223} | Test: {0: 240, 1: 226}

===== 10-Fold CV on TRAIN (with SMOTE) =====
       CV_Acc_mean  CV_Acc_std  CV_Prec_mean  CV_Prec_std  CV_Rec_mean  \
Model                                                                    
ADA         0.9957      0.0052        0.9956       0.0088       0.9956   
DT          0.9968      0.0049        0.9956       0.0088       0.9978   
KNN         0.9979      0.0043        1.0000       0.0000       0.9955   
RF          0.9957      0.0053        0.9934       0.0101       0.9978   
SVM         0.9968      0.0049        0.9978       0.0067       0.9955   

       CV_Rec_std  CV_F1_mean  CV_F1_std  
Model                                     
ADA        0.0089      0.9955     0.0055  
DT         0.0067      0.9967     0.0051  
KNN        0.0090      0.9977     0.0045  
RF         0.0067      0.9955     0.0055  
SVM        0.00